# CASME Spotting — process_all_datasets logic

CASME frame-folder spotting. ROI: mata+alis kanan, mata+alis kiri, mulut. Output mengikuti logika `process_all_datasets.ipynb`: metadata, event clips, event signal xlsx, plot.

In [ ]:
import gc
import re
import shutil
import sys
from dataclasses import dataclass
from pathlib import Path

import cv2
import dlib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from apex import ApexPhase, ApexSmoother, ApexPhaseVisualizer
from features_extraction.poc import POC
from features_extraction.quadran import Quadran
from features_extraction.vektor import Vektor
from spotting import SpottingConfig


In [ ]:
config = SpottingConfig(
    dataset_root=PROJECT_ROOT / 'dataset' / 'casme',
    output_root=PROJECT_ROOT / 'output' / 'casme_process_all',
    predictor_path=Path('./models/shape_predictor_68_face_landmarks.dat'),
    video_extensions={'.jpg', '.jpeg', '.png'},
    overwrite=True,
    regions={
        'mata_alis_kanan': list(range(42, 48)) + list(range(22, 27)),
        'mata_alis_kiri': list(range(36, 42)) + list(range(17, 22)),
        'mulut': list(range(48, 68)),
    },
    target_size={
        'mata_alis_kanan': (64, 40),
        'mata_alis_kiri': (64, 40),
        'mulut': (70, 35),
    },
)

SEGMENT_NAME = 'casme_all'

print(f'DATASET_ROOT = {config.dataset_root}')
print(f'OUTPUT_ROOT = {config.output_root}')
print(f'METADATA_PATH = {config.metadata_path}')
print(f"EVENT_SIGNAL_ROOT = {config.output_root / 'event_signals'}")
print(f'ROI = {list(config.regions)}')


In [ ]:
@dataclass(slots=True)
class FrameSample:
    name: str
    frame_paths: list[Path]

    @property
    def relative_key(self) -> Path:
        return Path(self.name)


def natural_sort_key(value: str):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', value)]


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def discover_frame_sample(dataset_root: Path) -> FrameSample:
    frame_paths = sorted(
        [p for p in dataset_root.iterdir() if p.is_file() and p.suffix.lower() in config.video_extensions],
        key=lambda p: natural_sort_key(p.name),
    )
    return FrameSample(name=SEGMENT_NAME, frame_paths=frame_paths)


def extract_region(image: np.ndarray, landmarks, indices: list[int]) -> np.ndarray:
    pts = [(landmarks.part(i).x, landmarks.part(i).y) for i in indices]
    xs, ys = zip(*pts)
    left = max(0, min(xs) - config.padding_x)
    top = max(0, min(ys) - config.padding_y)
    right = min(image.shape[1], max(xs) + config.padding_x)
    bottom = min(image.shape[0], max(ys) + config.padding_y)
    return image[top:bottom, left:right]


def detect_landmarks(gray: np.ndarray, detector, predictor):
    faces = detector(gray)
    if len(faces) > 0:
        return predictor(gray, faces[0])
    return None


def compute_frame_magnitudes(prev_gray, prev_lm, curr_gray, curr_lm) -> dict[str, float]:
    roi_values = {}
    for region_name, lm_indices in config.regions.items():
        try:
            roi_prev = extract_region(prev_gray, prev_lm, lm_indices)
            roi_curr = extract_region(curr_gray, curr_lm, lm_indices)
            if roi_prev.size == 0 or roi_curr.size == 0:
                roi_values[region_name] = 0.0
                continue
            target = config.target_size[region_name]
            roi_prev = cv2.resize(roi_prev, target)
            roi_curr = cv2.resize(roi_curr, target)
            poc = POC(roi_prev, roi_curr, config.block_size)
            vec = Vektor(poc.getPOC(), config.block_size)
            quad_data = Quadran(vec.getVektor()).getQuadran()
            magnitudes = [float(block[4]) for block in quad_data]
            roi_values[region_name] = float(np.mean(magnitudes)) if magnitudes else 0.0
        except Exception:
            roi_values[region_name] = 0.0
    return roi_values


def load_frames(frame_paths: list[Path]) -> list[np.ndarray]:
    frames = []
    for frame_path in frame_paths:
        frame = cv2.imread(str(frame_path))
        if frame is not None:
            frames.append(frame)
    return frames


def build_magnitude_signal(frames: list[np.ndarray], detector, predictor) -> tuple[list[float], pd.DataFrame]:
    if len(frames) < 3:
        return [], pd.DataFrame()
    grays = [cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY) for frame in frames]
    magnitudes = []
    roi_rows = []
    prev_gray = grays[0]
    prev_lm = detect_landmarks(prev_gray, detector, predictor)
    for signal_index, curr_gray in enumerate(grays[1:]):
        curr_lm = detect_landmarks(curr_gray, detector, predictor)
        if prev_lm is not None and curr_lm is not None:
            roi_values = compute_frame_magnitudes(prev_gray, prev_lm, curr_gray, curr_lm)
        else:
            roi_values = {region_name: 0.0 for region_name in config.regions}
        raw_value = float(np.mean(list(roi_values.values()))) if roi_values else 0.0
        magnitudes.append(raw_value)
        roi_rows.append({
            'signal_index': signal_index,
            'frame_index': signal_index + 1,
            **roi_values,
            'raw_magnitude': raw_value,
        })
        prev_gray = curr_gray
        prev_lm = curr_lm
    return magnitudes, pd.DataFrame(roi_rows)


def detect_events(magnitudes: list[float]) -> tuple[list[float], list[dict], dict]:
    smoothed = ApexSmoother.smooth(magnitudes)
    signal_arr = np.asarray(smoothed, dtype=float)
    height_threshold = float(signal_arr.mean() + signal_arr.std())
    detector = ApexPhase(
        distance_threshold=config.distance_threshold,
        prominence_threshold=config.prominence_threshold,
        cutoff_ratio=config.cutoff_ratio,
    )
    apex_indices = detector.find_top_k_apex(smoothed, k=config.top_k_apex, height=height_threshold)
    phases = detector.find_phase(smoothed, apex_indices) if apex_indices else {}
    events = []
    for event_no, apex_idx in enumerate(apex_indices, start=1):
        phase = phases[apex_idx]
        duration = int(phase['end'] - phase['start'])
        if duration < config.min_phase_duration:
            continue
        events.append({
            'event_no': event_no,
            'onset_signal': int(phase['start']),
            'apex_signal': int(apex_idx),
            'offset_signal': int(phase['end']),
            'duration': duration,
        })
    window_length = ApexSmoother.calculate_window_length(len(magnitudes))
    meta = {
        'window_length': window_length,
        'polyorder': ApexSmoother.calculate_polyorder(window_length),
        'height_threshold': height_threshold,
    }
    return smoothed, events, meta


def signal_index_to_frame_index(signal_index: int) -> int:
    return int(signal_index + 1)


In [ ]:
def save_event_clip_frames(frames: list[np.ndarray], sample_output_dir: Path, events: list[dict]) -> list[dict]:
    ensure_dir(sample_output_dir)
    rows = []
    for event in events:
        onset_frame = min(signal_index_to_frame_index(event['onset_signal']), len(frames) - 1)
        apex_frame = min(signal_index_to_frame_index(event['apex_signal']), len(frames) - 1)
        offset_frame = min(signal_index_to_frame_index(event['offset_signal']), len(frames) - 1)
        clip_dir = sample_output_dir / f'event_{onset_frame:05d}-{offset_frame:05d}'
        ensure_dir(clip_dir)
        for write_idx, frame_idx in enumerate(range(onset_frame, offset_frame + 1)):
            if frame_idx == apex_frame:
                out_path = clip_dir / f'frame_{write_idx:05d}-apex.png'
            else:
                out_path = clip_dir / f'frame_{write_idx:05d}.jpg'
            cv2.imwrite(str(out_path), frames[frame_idx])
        rows.append({
            'relative_sample': str(sample_output_dir.relative_to(config.output_dataset_root)),
            'event_no': event['event_no'],
            'onset_signal': event['onset_signal'],
            'apex_signal': event['apex_signal'],
            'offset_signal': event['offset_signal'],
            'onset_frame': onset_frame,
            'apex_frame': apex_frame,
            'offset_frame': offset_frame,
            'duration': event['duration'],
            'saved_frames': offset_frame - onset_frame + 1,
            'clip_dir': str(clip_dir.relative_to(config.output_root)),
        })
    return rows


def process_sample(sample: FrameSample) -> dict:
    detector = dlib.get_frontal_face_detector()
    predictor = dlib.shape_predictor(str(config.predictor_path))
    relative_key = sample.relative_key
    sample_output_dir = config.output_dataset_root / relative_key

    if sample_output_dir.exists() and config.overwrite:
        shutil.rmtree(sample_output_dir)
    elif sample_output_dir.exists() and not config.overwrite:
        return {'status': 'skip', 'relative_key': str(relative_key), 'rows': [], 'reason': 'exists'}

    frames = load_frames(sample.frame_paths)
    if len(frames) < 3:
        return {'status': 'skip', 'relative_key': str(relative_key), 'rows': [], 'reason': f'too few frames ({len(frames)})'}

    magnitudes, roi_df = build_magnitude_signal(frames, detector, predictor)
    if len(magnitudes) < 5:
        return {'status': 'skip', 'relative_key': str(relative_key), 'rows': [], 'reason': f'too few magnitudes ({len(magnitudes)})'}

    smoothed, events, meta = detect_events(magnitudes)
    rows = save_event_clip_frames(frames, sample_output_dir, events)

    metadata_rows = []
    if not rows:
        metadata_rows.append({
            'relative_sample': str(relative_key),
            'source_frames': str(config.dataset_root),
            'event_no': None,
            'onset_signal': None,
            'apex_signal': None,
            'offset_signal': None,
            'onset_frame': None,
            'apex_frame': None,
            'offset_frame': None,
            'duration': None,
            'saved_frames': 0,
            'window_length': meta['window_length'],
            'polyorder': meta['polyorder'],
            'height_threshold': meta['height_threshold'],
            'clip_dir': None,
        })
    else:
        for row in rows:
            metadata_rows.append({
                **row,
                'source_frames': str(config.dataset_root),
                'window_length': meta['window_length'],
                'polyorder': meta['polyorder'],
                'height_threshold': meta['height_threshold'],
            })

    del frames
    gc.collect()
    return {
        'status': 'processed',
        'relative_key': str(relative_key),
        'rows': metadata_rows,
        'magnitudes': magnitudes,
        'smoothed': smoothed,
        'events': events,
        'meta': meta,
        'roi_df': roi_df,
    }


In [ ]:
if not config.dataset_root.exists():
    raise FileNotFoundError(f'Dataset root not found: {config.dataset_root}')
if not config.predictor_path.exists():
    raise FileNotFoundError(f'Predictor not found: {config.predictor_path}')

ensure_dir(config.output_root)
ensure_dir(config.output_dataset_root)
ensure_dir(config.output_root / 'event_signals')

sample = discover_frame_sample(config.dataset_root)
print(f'Found {len(sample.frame_paths)} CASME frames')
print(sample.frame_paths[:5])


In [ ]:
result = process_sample(sample)
if result['status'] == 'skip':
    print(f"skip {result['relative_key']}: {result['reason']}")
else:
    print(f"processed {result['relative_key']}: {len(result['events'])} events")

metadata_df = pd.DataFrame(result['rows'])
metadata_df.to_excel(config.metadata_path, index=False)
print(f'Saved metadata: {config.metadata_path}')
metadata_df


In [ ]:
event_signal_root = config.output_root / 'event_signals'
relative_key = sample.relative_key
signal_path = event_signal_root / relative_key.with_suffix('.xlsx')
plot_path = event_signal_root / relative_key.with_suffix('.png')

magnitudes = result['magnitudes']
smoothed = result['smoothed']
events = result['events']
meta = result['meta']
roi_df = result['roi_df'].copy()

event_by_apex = {event['apex_signal']: event for event in events}
phases = {
    event['apex_signal']: {'start': event['onset_signal'], 'end': event['offset_signal']}
    for event in events
}
apex_indices = list(event_by_apex)

rows = []
for idx, (raw_value, smooth_value) in enumerate(zip(magnitudes, smoothed)):
    event = event_by_apex.get(idx)
    marker = ''
    event_no = None
    if event is not None:
        marker = 'apex'
        event_no = event['event_no']
    else:
        for candidate in events:
            if idx == candidate['onset_signal']:
                marker = 'onset'
                event_no = candidate['event_no']
                break
            if idx == candidate['offset_signal']:
                marker = 'offset'
                event_no = candidate['event_no']
                break
    roi_row = roi_df.iloc[idx].to_dict() if idx < len(roi_df) else {}
    rows.append({
        'relative_sample': str(relative_key),
        'source_frames': str(config.dataset_root),
        'signal_index': idx,
        'frame_index': signal_index_to_frame_index(idx),
        **{k: roi_row.get(k) for k in config.regions},
        'raw_magnitude': raw_value,
        'smoothed_magnitude': smooth_value,
        'height_threshold': meta['height_threshold'],
        'event_marker': marker,
        'event_no': event_no,
    })

signal_path.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(rows).to_excel(signal_path, index=False)
print(f'Saved event signal: {signal_path}')

visualizer = ApexPhaseVisualizer()
if apex_indices:
    visualizer.plot_phases(
        signal=smoothed,
        apex_indices=apex_indices,
        phases=phases,
        title=f'CASME Micro-Expression Spotting: {relative_key}',
        save_path=str(plot_path),
    )
    print(f'Saved plot: {plot_path}')
else:
    print('No apex indices; plot skipped')
